# Phase 1 -> Phase 2: tokenize the corpora into training binariesRun this **once** on Kaggle. It downloads the Phase 1 processed corpora fromGoogle Drive, encodes them with the Phase 1 tokenizers, and writes compact`uint16` binaries that the training runs memory-map.Why binaries rather than training straight from the JSONL:* **Size.** The vocabulary is 8,000, so a token id fits in 16 bits. 11.5 GB of  JSONL becomes about 2.4 GB of `uint16` - roughly a 5x reduction.* **Speed.** Training slices random windows out of a memory-mapped array. No JSON  parsing, no tokenizer call, no Python in the data path.* **Reuse.** Save the output as a Kaggle Dataset and every later training session  mounts it in seconds. Sessions are capped at 12 hours, so training will span  several of them and re-tokenizing each time would be wasteful.**Runtime: CPU is fine** (no GPU needed here - don't spend GPU quota on this).

In [ ]:
!pip install -q gdown sentencepieceimport os, json, time, gzip, shutilfrom pathlib import Pathimport numpy as npprint("ready")

## 1. Fetch the corpora and tokenizersSet `DRIVE_FOLDER` to your shared Phase 1 folder. It must be shared as**"Anyone with the link -> Viewer"**, which is what the Phase 1 README alreadyrequires.`gdown --folder` walks the folder and pulls everything. If Drive throws itsvirus-scan interstitial on the two large train files, the `--no-cookies` retrybelow handles it.

In [ ]:
DRIVE_FOLDER = "https://drive.google.com/drive/folders/1ggiw-RzYTUgkXLhky2u73tObkw7MKbJT"WORK = Path("/kaggle/working")RAW  = WORK / "corpora"RAW.mkdir(parents=True, exist_ok=True)# --folder is capped at 50 files, which is ample here.!gdown --folder "{DRIVE_FOLDER}" -O {RAW} --remaining-okfor p in sorted(RAW.rglob("*")):    if p.is_file():        print(f"{p.stat().st_size/1e9:7.2f} GB  {p.relative_to(RAW)}")

If any file is missing or truncated, download it individually by its ownfile id. Right-click the file in Drive -> Share -> Copy link, and take the idbetween `/d/` and `/view`.

In [ ]:
# Fallback for a single large file that the folder download truncated.# FILE_ID = "..."# !gdown --id {FILE_ID} -O {RAW}/hindi/hi_train.jsonl

## 2. Locate the splits and tokenizersThe tokenizer `.model` files are small and live in the GitHub repository, not onDrive, so this clones the repo for them. Replace the URL if yours differs.

In [ ]:
REPO = "https://github.com/CL3-410/individual-project-Karan-thayat.git"if not Path("/kaggle/working/repo").exists():    !git clone -q --branch phase-1 {REPO} /kaggle/working/repoTOK = Path("/kaggle/working/repo")def find(pattern, root):    hits = sorted(root.rglob(pattern))    if not hits:        raise FileNotFoundError(f"{pattern} not found under {root}")    return hits[0]LANGS = {    "hi": {        "tokenizer": find("hi_unigram_8000.model", TOK),        "splits": {s: find(f"hi_{s}.jsonl", RAW) for s in ("train", "validation", "test")},    },    "ne": {        "tokenizer": find("ne_unigram_8000.model", TOK),        "splits": {s: find(f"ne_{s}.jsonl", RAW) for s in ("train", "validation", "test")},    },}for lang, spec in LANGS.items():    print(lang, spec["tokenizer"].name)    for s, p in spec["splits"].items():        print(f"   {s:11s} {p.stat().st_size/1e9:6.2f} GB")

## 3. Encode to `uint16`Documents are concatenated into one long stream with an EOS separator betweenthem, which is the standard arrangement for causal LM pretraining: the modellearns document boundaries as an ordinary token, and training windows can startanywhere without tracking document structure.The write is streamed in chunks so peak memory stays flat regardless of corpussize.

In [ ]:
import sentencepiece as spmCHUNK_DOCS = 20_000   # documents buffered before each flushdef encode_split(jsonl_path: Path, sp: spm.SentencePieceProcessor, out_path: Path) -> dict:    """Encode one split to a flat uint16 array on disk. Returns a manifest."""    eos = sp.eos_id()    assert sp.get_piece_size() <= 65535, "uint16 requires vocab <= 65535"    n_docs = n_tokens = 0    buf: list[int] = []    t0 = time.time()    with open(out_path, "wb") as out, open(jsonl_path, encoding="utf-8", errors="replace") as f:        for line in f:            try:                text = json.loads(line).get("text", "")            except Exception:                continue            if not text:                continue            ids = sp.encode(text)            buf.extend(ids)            buf.append(eos)            n_docs += 1            n_tokens += len(ids) + 1            if n_docs % CHUNK_DOCS == 0:                np.asarray(buf, dtype=np.uint16).tofile(out)                buf.clear()                rate = n_tokens / max(time.time() - t0, 1e-9)                print(f"   {n_docs:>9,} docs  {n_tokens/1e6:>8.1f}M tokens  {rate/1e6:.2f}M tok/s", flush=True)        if buf:            np.asarray(buf, dtype=np.uint16).tofile(out)    return {        "source": str(jsonl_path),        "documents": n_docs,        "tokens": n_tokens,        "bytes": out_path.stat().st_size,        "seconds": round(time.time() - t0, 1),    }OUT = WORK / "tokenized"OUT.mkdir(exist_ok=True)manifest = {}for lang, spec in LANGS.items():    sp = spm.SentencePieceProcessor(model_file=str(spec["tokenizer"]))    manifest[lang] = {"vocab_size": sp.get_piece_size(), "splits": {}}    for split, path in spec["splits"].items():        print(f"\n[{lang}/{split}]")        out_path = OUT / f"{lang}_{split}.bin"        manifest[lang]["splits"][split] = encode_split(path, sp, out_path)        m = manifest[lang]["splits"][split]        print(f"   done: {m['tokens']:,} tokens -> {m['bytes']/1e9:.2f} GB")(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))print("\nwrote", OUT / "manifest.json")

## 4. Verify before trusting itTwo checks worth doing now rather than discovering mid-training: that the tokencounts match Phase 1, and that decoding a window returns readable text in theright language.

In [ ]:
EXPECTED = {"hi": 578_319_564, "ne": 610_989_716}   # Phase 1 reported train tokensfor lang, spec in manifest.items():    got = spec["splits"]["train"]["tokens"]    want = EXPECTED[lang]    # The binary adds one EOS per document, so it runs slightly above Phase 1.    delta = 100 * (got - want) / want    print(f"{lang}: {got:,} train tokens vs Phase 1 {want:,}  ({delta:+.2f}%)")print()for lang, spec in LANGS.items():    sp = spm.SentencePieceProcessor(model_file=str(spec["tokenizer"]))    arr = np.memmap(OUT / f"{lang}_train.bin", dtype=np.uint16, mode="r")    print(f"[{lang}] {len(arr):,} tokens on disk")    print("   ", sp.decode([int(t) for t in arr[10_000:10_060]])[:200])    print()

## 5. Save as a Kaggle DatasetThe binaries in `/kaggle/working/tokenized/` are the artifact. Save this notebookversion, then from the notebook's **Output** tab choose *Create Dataset* - or use**+ Add Data -> Upload** and point at that folder.Name it something like `lma-phase2-tokenized`. Every training session then mountsit read-only at `/kaggle/input/lma-phase2-tokenized/`, in seconds.Delete the raw corpora first so the output stays small.

In [ ]:
shutil.rmtree(RAW, ignore_errors=True)shutil.rmtree("/kaggle/working/repo", ignore_errors=True)total = 0for p in sorted(OUT.iterdir()):    total += p.stat().st_size    print(f"{p.stat().st_size/1e9:7.2f} GB  {p.name}")print(f"\n{total/1e9:.2f} GB total - save this as a Kaggle Dataset")